# Course Bootstrap

[Colab で開く](https://colab.research.google.com/github/henry753951/basketball-hackathon-course/blob/main/init_colab_ja.ipynb)

[English](init_colab_en.ipynb) | [Japanese](init_colab_ja.ipynb) | [Traditional Chinese](init_colab.ipynb)

## Google Drive に教材をセットアップする

### Colab のランタイム設定

実行前に Colab の **ランタイム → ランタイムのタイプを変更** を開き、ハードウェア アクセラレータを
**T4 GPU** に設定する。別の NVIDIA GPU が割り当てられた場合も利用できる。**TPU は選択しない**。
本教材の Ultralytics／PyTorch の処理では TPU を直接利用できず、CPU での実行になるためである。

1. 実行環境を確認し、Google Drive をマウントする。
2. 同期方法を選ぶ。
   - `fresh_install`: 教材リポジトリ全体を再インストールする。
   - `update_preserve_assets`: 教材を更新し、`assets/` に追加した学生のデータを保持する。
3. GitHub から最新のスナップショットを取得し、Google Drive に同期する。
4. 教材フォルダの内容を確認し、Colab に必要なパッケージをインストールする。
5. Day 4 の三つのモデルがそろっていることと、Google Drive の `assets/results/` に書き込めることを確認する。

> 更新時には Notebook と `src/` のコードが上書きされる。教材を直接編集した場合は、先に別名で保存する。


In [1]:
import os
os.environ["BASKETBALL_COURSE_LANGUAGE"] = "ja"

from pathlib import Path

try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
except ModuleNotFoundError:
    drive = None
    IN_COLAB = False

COURSE_TITLE = "Python によるバスケットボール映像分析"
DRIVE_MOUNTED = False

if IN_COLAB and drive is not None:
    try:
        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except NotImplementedError:
        print("この Colab ランタイムでは Drive をマウントできないため、ローカルの /content を使います。")

if DRIVE_MOUNTED:
    ROOT = Path("/content/drive/MyDrive/basketball_hackathon")
    TARGET_DIR = ROOT / "course"
elif IN_COLAB:
    ROOT = Path("/content/basketball_hackathon")
    TARGET_DIR = ROOT / "course"
else:
    ROOT = Path.cwd().resolve()
    TARGET_DIR = ROOT
    print("ローカル確認：現在の教材リポジトリを使います。")

ROOT.mkdir(parents=True, exist_ok=True)

print(COURSE_TITLE)
print("Drive mounted:", DRIVE_MOUNTED)
print("作業フォルダ：", ROOT)
print("教材フォルダ：", TARGET_DIR)


ローカル確認：現在の教材リポジトリを使います。
Python によるバスケットボール映像分析
Drive mounted: False
作業フォルダ： /Users/henry753951/Developer/sandbox/basketball-hackathon-course
教材フォルダ： /Users/henry753951/Developer/sandbox/basketball-hackathon-course


In [2]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/henry753951/basketball-hackathon-course.git"
REPO_BRANCH = "main"
SYNC_MODE = "update_preserve_assets"  # "fresh_install" も選択できます。
PRESERVE_PATHS = ("assets",)

if IN_COLAB:
    LOCAL_CLONE_DIR = Path("/content/basketball_hackathon_course_tmp")
    if LOCAL_CLONE_DIR.exists():
        shutil.rmtree(LOCAL_CLONE_DIR)

    cmd = [
        "git",
        "clone",
        "--depth",
        "1",
        "-b",
        REPO_BRANCH,
        REPO_URL,
        str(LOCAL_CLONE_DIR),
    ]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    LOCAL_CLONE_DIR = TARGET_DIR
    print("ローカル確認：git clone を省略します。")

if str(LOCAL_CLONE_DIR) not in sys.path:
    sys.path.insert(0, str(LOCAL_CLONE_DIR))

from src.course_setup import replace_tree_from_snapshot, rsync_tree  # noqa: E402

print("取得元フォルダ：", LOCAL_CLONE_DIR)
print("同期方法：", SYNC_MODE)
if SYNC_MODE == "update_preserve_assets":
    print("保持するパス：", ", ".join(PRESERVE_PATHS))
print("assets は統合され、同名ファイルには新版を優先します。Notebook と src は更新されます。")


ローカル確認：git clone を省略します。
取得元フォルダ： /Users/henry753951/Developer/sandbox/basketball-hackathon-course
同期方法： update_preserve_assets
保持するパス： assets
assets は統合され、同名ファイルには新版を優先します。Notebook と src は更新されます。


In [3]:
import shutil

if IN_COLAB:
    if SYNC_MODE == "fresh_install":
        if TARGET_DIR.exists():
            shutil.rmtree(TARGET_DIR)
        rsync_tree(LOCAL_CLONE_DIR, TARGET_DIR)
    elif SYNC_MODE == "update_preserve_assets":
        replace_tree_from_snapshot(
            LOCAL_CLONE_DIR,
            TARGET_DIR,
            preserve_paths=PRESERVE_PATHS,
        )
    else:
        raise ValueError(f"未対応の SYNC_MODE：{SYNC_MODE}")
else:
    print("ローカル確認：教材は現在のリポジトリにあります。")

print("教材の準備完了：", TARGET_DIR)


ローカル確認：教材は現在のリポジトリにあります。
教材の準備完了： /Users/henry753951/Developer/sandbox/basketball-hackathon-course


In [4]:
import subprocess
import sys

requirements = TARGET_DIR / "requirements.txt"
if IN_COLAB and requirements.exists():
    print("Colab の教材パッケージを確認しています。")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
elif IN_COLAB:
    print("requirements.txt がありません。リポジトリの内容を確認してください。")
else:
    print("ローカル確認：パッケージは uv sync で管理するため、pip install を省略します。")

print("\n教材フォルダの内容：")
for p in sorted(TARGET_DIR.iterdir()):
    print("-", p.name)


ローカル確認：パッケージは uv sync で管理するため、pip install を省略します。

教材フォルダの内容：
- .DS_Store
- .git
- .gitattributes
- .gitignore
- .ruff_cache
- .tmp
- .venv
- .vscode
- PROJECT_PLAN.md
- README.md
- assets
- day1
- day2
- day3
- day4
- day5
- init_colab.ipynb
- init_colab_en.ipynb
- init_colab_ja.ipynb
- output
- pyproject.toml
- python_basketball_hackathon_course_material.tex
- python_basketball_hackathon_course_material_en.tex
- python_basketball_hackathon_course_material_ja.tex
- requirements.txt
- scripts
- slides
- src
- test
- tests
- uv.lock
- workshop_schedule_updated.tex
- workshop_schedule_updated_en.tex
- workshop_schedule_updated_ja.tex


In [5]:
from src.course_setup import prepare_day4_workspace

DAY4_MODELS, DAY4_RESULTS = prepare_day4_workspace(TARGET_DIR)
print("\nDay 4 のモデル確認：")
for model_name, model_path in DAY4_MODELS.items():
    size_mb = model_path.stat().st_size / (1024 * 1024)
    print(f"- {model_name}: {model_path} ({size_mb:.1f} MB)")
print("Day 4 の結果フォルダ：", DAY4_RESULTS)
print("✅ Day 4 のモデルがそろい、結果の書き込み確認に成功しました。")



Day 4 のモデル確認：
- ball_rim: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/models/detectors/ball_rimV8.pt (6.0 MB)
- shot_detection: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/models/detectors/shot_detection.pt (5.9 MB)
- yolo_pose: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/models/pose/yolov8n-pose.pt (6.5 MB)
Day 4 の結果フォルダ： /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results
✅ Day 4 のモデルがそろい、結果の書き込み確認に成功しました。


In [6]:
if IN_COLAB:
    # Google Drive のパス：
    drive_path = Path("/content/drive/MyDrive/basketball_hackathon")
    if drive_path.exists():
        print(f"教材フォルダを {drive_path} に作成しました。")
        print("https://drive.google.com")
    